# Multicomponent diffusion in garnet — 1D planar tutorial

**Companion to `MC_diffusion-garnet_example.ipynb` (2D): the same benchmark, solved with the pure-Python 1D solver.**

This notebook reproduces the coupled three-component (Mg–Fe–Ca) garnet diffusion
benchmark of Vielzeuf & Saul (2011) with `UWDiffusion.fd_1d.MulticomponentDiffusion1D`,
a conservative vertex-centered finite-volume solver.

It needs only `numpy`, `scipy` and `matplotlib` — **no Underworld3, PETSc or MPI** —
and runs in well under a second.

### The benchmark

Initial conditions and reference data follow [Vielzeuf & Saul, 2011](https://doi.org/10.1007/s00410-010-0557-4)
(Figs. 6 and 9). The multicomponent coupling is outlined in:

- [Gaidies et al., 2008](https://doi.org/10.1007/s00410-007-0263-z)
- [Perchuk & Gerya, 2005](https://citeseerx.ist.psu.edu/document?repid=rep1&type=pdf&doi=973b9a24a4dac26a8631800752cc2dd8b0676551)
- [Faryad et al., 2022](https://doi.org/10.1093/petrology/egac118)
- [Florence & Spear, 1991](https://link.springer.com/article/10.1007/BF00310683)
- [Tirone & Ganguly, 2010](https://www.sciencedirect.com/science/article/pii/S1342937X1000002X)

### Governing equations

Fick's first law, expanded for a multicomponent system:

$$
J_i = -\sum_{j=1}^{N} D_{ij}\,\frac{\partial X_j}{\partial x}
$$

where $X_j$ are mole fractions and $D_{ij}$ the interdiffusion matrix entries.
Because the mole fractions must sum to one, only $N-1$ components are solved as
independent unknowns and the last one (the *dependent component*) is recovered
from the closure relation

$$
X_{\mathrm{Ca}} = 1 - X_{\mathrm{Mg}} - X_{\mathrm{Fe}}.
$$

Substituting the closure into the fluxes turns the full $N \times N$ matrix into
a reduced $(N-1)\times(N-1)$ interdiffusion matrix acting on the independent
components (Mg and Fe here). `MulticomponentDiffusion1D` solves

$$
\frac{\partial X_i}{\partial t}
= \frac{1}{r^{\alpha}}\frac{\partial}{\partial r}
\left(r^{\alpha}\sum_j D_{ij}\frac{\partial X_j}{\partial r}\right),
\qquad \alpha = 0 \;\text{(planar)},
$$

using an unconditionally stable implicit (backward-Euler) finite-volume scheme
with zero-flux Neumann boundaries.

### Geometry, units, and why 1D is equivalent to the 2D slab here

The 2D notebook solves a $12\,\mu\text{m} \times 1\,\mu\text{m}$ box. Its
initial condition varies with $x$ only, and no diffusive flux crosses the
$y$-normal faces, so the solution stays uniform in $y$:
$\partial/\partial y \equiv 0$ everywhere, for all time. The 2D slab problem
therefore reduces **exactly** to the 1D planar problem ($\alpha = 0$) solved
here — the $1\,\mu\text{m}$ thickness carries no information, and the 1D
solution is the $y$-midplane profile of the 2D run at a fraction of the cost.

**Units.** The 1D solver is unit-agnostic; this notebook uses SI throughout:

| quantity | unit |
|---|---|
| distance $x$ | m (grid built in $\mu$m, converted with `1.0e-6`) |
| time $t$ | s |
| diffusivity $D$ | m² s⁻¹ |
| composition $X$ | mole fraction (dimensionless, $\sum_k X_k = 1$) |

**Boundary conditions.** Both domain ends are closed (zero-flux Neumann),
matching the natural boundaries of the 2D model, so the total amount of each
component is conserved.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import matplotlib.pyplot as plt

try:
    from UWDiffusion.fd_1d import MulticomponentDiffusion1D
except ImportError:
    # UWDiffusion is not installed as a package: fall back to the in-repo
    # source tree (src/UWDiffusion) by walking up from the current directory.
    _repo_src = None
    for _base in (Path.cwd(), *Path.cwd().parents):
        if (_base / "src" / "UWDiffusion" / "fd_1d.py").exists():
            _repo_src = _base / "src"
            break
    if _repo_src is None:
        raise ImportError(
            "UWDiffusion is not installed and no src/UWDiffusion/fd_1d.py "
            "was found above the current directory."
        )
    sys.path.insert(0, str(_repo_src))
    from UWDiffusion.fd_1d import MulticomponentDiffusion1D


In [ ]:
# --- User-tunable parameters --------------------------------------------------
QUICK_RUN = False    # True -> coarser grid (faster, lower accuracy)

X_MAX_UM = 12.0      # domain length (µm), as in the 2D benchmark
STEP_UM = 6.0        # position of the initial compositional step (µm)
CELL_SIZE_UM = 0.1 if QUICK_RUN else 0.05   # grid spacing (µm)
DT_S = 5.0e3         # time step (s); implicit scheme, so stability is not a limit

DURATION_1 = 1.0e5     # stage 1: run to t = 1e5 s
DURATION_2 = 5.184e5   # stage 2: a further 5.184e5 s (total 6.184e5 s)

In [ ]:
# Grid in metres (SI); the diffusivity matrix below is in m^2/s
r_um = np.arange(0.0, X_MAX_UM + 0.5 * CELL_SIZE_UM, CELL_SIZE_UM)
r_m = r_um * 1.0e-6

# Reduced interdiffusion matrix (m^2/s) acting on the independent components
# (Mg, Fe) — benchmark values, identical to the 2D notebook:
#     D_MgMg =  0.66e-19,  D_MgFe = -4.74e-19
#     D_FeMg = -0.40e-19,  D_FeFe =  4.50e-19
D_reduced = np.array(
    [[ 0.66e-19, -4.74e-19],
     [-0.40e-19,  4.50e-19]]
)

model = MulticomponentDiffusion1D(
    grid=r_m,
    diffusivity=D_reduced,           # constant reduced 2x2 matrix
    geometry="planar",               # alpha = 0 (planar / slab)
    components=["Mg", "Fe", "Ca"],
    dependent_component="Ca",        # closure: Ca = 1 - Mg - Fe
    bcs={"left": ("neumann", 0.0), "right": ("neumann", 0.0)},  # zero flux
)

print(f"grid: {model.n_points} vertices over {r_um[-1]:.2f} µm "
      f"({model.geometry}, alpha = {model.alpha})")
print(f"components: {model.components}; dependent: {model.dependent_component}")
print(f"independent components solved: {model.independent_components}")

### Initial conditions

A compositional step at $x = 6\,\mu$m, with values from Figs. 6 and 9 of the
benchmark:

$$
X_{\mathrm{Mg}} = \begin{cases} 0.461185 & x \le 6\,\mu\text{m}\\ 0.4569965 & \text{else} \end{cases}
\qquad
X_{\mathrm{Fe}} = \begin{cases} 0.5190967 & x \le 6\,\mu\text{m}\\ 0.50997849 & \text{else} \end{cases}
$$

$X_{\mathrm{Ca}}$ follows from closure and therefore inherits a larger step
($0.0197 \to 0.0330$). Only the Mg and Fe arrays are supplied below — the
solver computes Ca automatically.

In [ ]:
Mg0 = np.where(r_um <= STEP_UM, 0.461185, 0.4569965)
Fe0 = np.where(r_um <= STEP_UM, 0.5190967, 0.50997849)

model.set_initial_composition({"Mg": Mg0, "Fe": Fe0})

initial = {c: model.get_composition(c).copy() for c in ("Mg", "Fe", "Ca")}
initial_mass = model.total_mass()

fig, ax = plt.subplots(figsize=(8, 4.5))
for comp, colour in zip(("Mg", "Fe", "Ca"), ("tab:green", "tab:red", "gold")):
    ax.plot(r_um, initial[comp], color=colour, label=f"{comp} (t = 0 s)")
ax.set_xlabel("x (µm)")
ax.set_ylabel("mole fraction")
ax.set_title("Initial step profile")
ax.legend()
ax.grid(alpha=0.3)
plt.show()

In [ ]:
# --- Stage 1: 0 -> 1e5 s ------------------------------------------------------
result_stage1 = model.run_to_time(DURATION_1, DT_S)
stage1 = {c: model.get_composition(c).copy() for c in ("Mg", "Fe", "Ca")}
print(f"stage 1 complete: model time = {model.time:.3g} s")

fig, ax = plt.subplots(figsize=(8, 4.5))
for comp, colour in zip(("Mg", "Fe", "Ca"), ("tab:green", "tab:red", "gold")):
    ax.plot(r_um, initial[comp], color=colour, ls=":", alpha=0.7, label=f"{comp} (t = 0 s)")
    ax.plot(r_um, stage1[comp], color=colour, label=f"{comp} (t = {DURATION_1:.3g} s)")
ax.set_xlabel("x (µm)")
ax.set_ylabel("mole fraction")
ax.set_xlim(3.0, 9.0)
ax.set_title("After stage 1 (zoom near the step)")
ax.legend(ncols=2)
ax.grid(alpha=0.3)
plt.show()

### Stage 2 — diffusion on to $t = 6.184 \times 10^5$ s

The second stage runs for a further $5.184 \times 10^5$ s (6 days), matching the
2D notebook. Because the scheme is implicit, the same time step can simply be
reused; `run_to_time` tracks the absolute model time across stages.

In [ ]:
# --- Stage 2: 1e5 s -> 1e5 + 5.184e5 s ----------------------------------------
result_stage2 = model.run_to_time(DURATION_1 + DURATION_2, DT_S)
stage2 = {c: model.get_composition(c).copy() for c in ("Mg", "Fe", "Ca")}
print(f"stage 2 complete: model time = {model.time:.4g} s")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4), sharey=True)
for ax, comp, colour in zip(axes, ("Mg", "Fe", "Ca"), ("tab:green", "tab:red", "gold")):
    ax.plot(r_um, initial[comp], color=colour, ls=":", label="t = 0 s")
    ax.plot(r_um, stage1[comp], color=colour, ls="--", label=f"t = {DURATION_1:.3g} s")
    ax.plot(r_um, stage2[comp], color=colour, label=f"t = {DURATION_1 + DURATION_2:.3g} s")
    ax.set_title(comp)
    ax.set_xlabel("x (µm)")
    ax.grid(alpha=0.3)
axes[0].set_ylabel("mole fraction")
axes[0].legend()
plt.tight_layout()
plt.show()

In [ ]:
# Reference data digitized from Fig. 6 of Vielzeuf & Saul (2011) — the same
# digitized arrays used in the 2D notebook: (x in µm, composition in mol %).
Mg_digitized = np.array([
    (3.0554956807897984, 46.12225405069445),
    (3.175767174002468, 46.122662966222165),
    (3.2960386672151376, 46.122662966222165),
    (3.4181324557795145, 46.124162323157115),
    (3.6627405626048164, 46.12351448501858),
    (3.777124640065817, 46.122662966222165),
    (3.8948729550992, 46.12351448501858),
    (4.017667626491156, 46.12552537491616),
    (4.127005347593583, 46.12582827530706),
    (4.258210612916496, 46.126615816323394),
    (4.378482106129166, 46.13056866642462),
    (4.4987535993418355, 46.137816471080534),
    (4.739296585767175, 46.157829701605515),
    (4.859568078979844, 46.172278050251386),
    (4.979839572192514, 46.191633385229814),
    (5.1001110654051836, 46.21616831689261),
    (5.220382558617853, 46.24561023488798),
    (5.329720279720281, 46.27859608745685),
    (5.4281242287124645, 46.31091555916575),
    (5.526528177704649, 46.345067578239586),
    (5.635865898807076, 46.37469123646949),
    (5.73426984779926, 46.37284138051078),
    (5.7927025915261225, 46.33482197251743),
    (5.835656696244932, 46.30044385993754),
    (5.85454134101193, 46.260303932850476),
    (5.865475113122173, 46.22611859473364),
    (5.887342657342659, 46.187135314424964),
    (5.902649938296998, 46.14635280579436),
    (5.922335115864529, 46.11645729446363),
    (5.935451254627726, 46.07318418736885),
    (5.952945290004115, 46.02201863196372),
    (5.971688899335959, 45.962231774182634),
    (5.98756890168655, 45.91425235226427),
    (6.004088852324147, 45.87177057244072),
    (6.009800904977377, 45.8281892641982),
    (6.05839325380502, 45.709770053475935),
    (6.029965446318388, 45.79430379746835),
    (6.031668449197862, 45.75891897380356),
    (6.073216783216784, 45.66266025857984),
    (6.089617441382148, 45.62750033845529),
    (6.1078403948992195, 45.59114093278278),
    (6.122418757712876, 45.55215765247411),
    (6.148191220544162, 45.51413824448076),
    (6.188021390374333, 45.47718980572666),
    (6.25909090909091, 45.447543429967446),
    (6.37936240230358, 45.45790262333618),
    (6.483233237350886, 45.49124627699181),
    (6.576170300287949, 45.52391976353257),
    (6.680041135335254, 45.557105530359436),
    (6.794845742492803, 45.58861928702855),
    (6.915117235705472, 45.61710706879258),
    (7.035388728918142, 45.63959742281681),
    (7.155660222130812, 45.65745340086029),
    (7.275931715343481, 45.67122022362664),
    (7.396203208556151, 45.68103419629176),
    (7.5164747017688205, 45.68812206543879),
    (7.63674619498149, 45.69343796729907),
    (7.757017688194161, 45.69670929152077),
    (7.87728918140683, 45.699162784687054),
    (7.9975606746195, 45.699844310566576),
    (8.11783216783217, 45.699980615742476),
    (8.238103661044839, 45.70270671926057),
    (8.358375154257509, 45.702843024436476),
    (8.478646647470178, 45.702843024436476),
    (8.598918140682848, 45.702843024436476),
    (8.719189633895517, 45.702843024436476),
    (8.839461127108187, 45.702843024436476),
    (8.959732620320857, 45.702843024436476),
    (9.041735911147677, 45.702843024436476),
    (4.624615384615385, 46.1493670886076),
    (3.516923076923077, 46.12405063291139)
])

Fe_digitized = np.array([
    (3.0882969971205263, 51.91246621815881),
    (3.2161380248710567, 51.91147022701102),
    (3.449111476758536, 51.91246621815881),
    (3.5693829699712056, 51.91246621815881),
    (3.6896544631838752, 51.91246621815881),
    (3.809925956396545, 51.91246621815881),
    (4.050468942821884, 51.909756823017126),
    (4.291011929247224, 51.904943204687896),
    (4.411283422459894, 51.90223380954621),
    (4.531554915672563, 51.897602364859566),
    (4.656746606334842, 51.88971231288214),
    (4.772097902097903, 51.876688307954474),
    (4.892369395310572, 51.86186768495721),
    (5.012640888523242, 51.84264718950763),
    (5.1329123817359115, 51.81891103548857),
    (5.242250102838339, 51.79231882391275),
    (5.340654051830523, 51.765147681751095),
    (5.4335911147675855, 51.7362252745673),
    (5.515594405594406, 51.707772903882955),
    (5.592130810366106, 51.678479016239926),
    (5.663200329082683, 51.64839668027523),
    (5.7233360756890175, 51.62054625955954),
    (5.778004936240231, 51.5920165602898),
    (5.827206910736323, 51.56380527284227),
    (5.869119703825587, 51.52994748241427),
    (5.923907856849034, 51.49709799608993),
    (5.958412176059236, 51.46541601978034),
    (6.0021472645002065, 51.430231513426484),
    (6.055113122171947, 51.397435095739176),
    (6.098848210612918, 51.366867560807314),
    (6.13881941587824, 51.3378693412849),
    (6.204422048539697, 51.305445920303605),
    (6.264557795146032, 51.27627939738945),
    (6.3301604278074874, 51.24783460793897),
    (6.401229946524065, 51.21942014342394),
    (6.483233237350886, 51.19014899948249),
    (6.570703414232827, 51.161014317750556),
    (6.669107363225011, 51.13273934793859),
    (6.78391197038256, 51.10370018975332),
    (6.9041834635952295, 51.07834303009393),
    (7.024454956807899, 51.05773310123836),
    (7.144726450020569, 51.042101975420934),
    (7.256493898258605, 51.030536228828446),
    (7.374335664335666, 51.02015819746543),
    (7.4946071575483355, 51.01442600927334),
    (7.614878650761005, 51.008199800314685),
    (7.735150143973675, 51.006208279099425),
    (7.855421637186344, 51.0049346318106),
    (7.975693130399014, 51.00308205393594),
    (8.095964623611684, 51.00308205393594),
    (8.216236116824355, 51.002387337232946),
    (8.336507610037025, 51.001345262178454),
    (8.456779103249694, 51.00053475935829),
    (8.577050596462364, 51.00053475935829),
    (8.697322089675033, 51.00053475935829),
    (8.817593582887703, 51.00053475935829),
    (8.937865076100373, 51.00053475935829),
    (9.025335252982313, 51.000789488816054),
    (3.332307692307692, 51.91182795698924),
    (3.9230769230769234, 51.91182795698924),
    (4.163076923076924, 51.909677419354836)
])

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(r_um, stage2["Mg"], color="tab:green", label="Mg (model)")
ax.plot(r_um, stage2["Fe"], color="tab:red", label="Fe (model)")
ax.plot(r_um, stage2["Ca"], color="gold", label="Ca (model)")

ax.scatter(Mg_digitized[:, 0], Mg_digitized[:, 1] / 100.0, marker="x", c="k",
           label="Mg (digitized: Vielzeuf & Saul 2011)")
ax.scatter(Fe_digitized[:, 0], Fe_digitized[:, 1] / 100.0, marker="+", c="k",
           label="Fe (digitized: Vielzeuf & Saul 2011)")

ax.set_xlim(3.0, 9.0)
ax.set_ylim(0.45, 0.54)
ax.set_xlabel("x (µm)")
ax.set_ylabel("mole fraction")
ax.set_title("Final profiles vs digitized benchmark data")
ax.legend(ncols=2, fontsize=9)
ax.grid(ls=":", alpha=0.5)
plt.show()

In [ ]:
# Checks: closure and mass conservation (zero-flux => component amounts conserved)
X_now = model.get_composition()
print(f"max |sum_k X_k - 1| = {np.max(np.abs(X_now.sum(axis=1) - 1.0)):.3e}")

mass_now = model.total_mass()
print("\ncomponent totals (V-weighted integrals):")
for comp in ("Mg", "Fe", "Ca"):
    drift = abs(mass_now[comp] - initial_mass[comp]) / abs(initial_mass[comp])
    print(f"  {comp}: initial = {initial_mass[comp]:.6e}, final = {mass_now[comp]:.6e}, "
          f"relative drift = {drift:.2e}")

### Summary

- The 2D slab benchmark reduces **exactly** to the 1D planar problem, because the
  initial condition and the boundary fluxes have no $y$ dependence.
- The constant reduced 2×2 interdiffusion matrix plus the Ca closure reproduces
  the coupled Mg–Fe–Ca diffusion of the 2D notebook, stage for stage, at a small
  fraction of the computational cost.
- Zero-flux boundaries keep the total amount of every component constant, and
  closure $\sum_k X_k = 1$ is maintained to machine precision.

**Things to try**

- Set `QUICK_RUN = True` and compare the coarser solution.
- Switch `geometry="cylindrical"` or `"spherical"` to see how curvature changes
  the relaxation of the step (the domain then behaves like a shell, not a slab).
- Replace the constant matrix with a composition-dependent callable using
  `UWDiffusion.fd_1d.darken_reduced_diffusivity` (see the CZGM 1D tutorial).